## Step 4
Inputs: 
Output: New 

In [ ]:
import glob
import os
import time
import dask.dataframe as dd
import pandas as pd
from datetime import datetime

def load_valid_parquet_files(folder_pattern):
    """
    Load all valid parquet files matching the given pattern into a Dask DataFrame.
    Excludes hidden files (like macOS ._* files).
    
    Args:
        folder_pattern: Glob pattern to match parquet files
        
    Returns:
        Dask DataFrame containing the data from all valid parquet files
    """
    # Get all files matching the pattern
    all_files = glob.glob(folder_pattern)
    
    # Filter out hidden files (like macOS ._ files)
    valid_files = [f for f in all_files if not os.path.basename(f).startswith('._')]
    
    # Sort files to ensure consistent reading order
    valid_files = sorted(valid_files)
    
    print(f"Found {len(valid_files)} valid parquet files from pattern: {folder_pattern}")
    
    if not valid_files:
        raise ValueError(f"No valid parquet files found for pattern: {folder_pattern}")
    
    # Load files into a Dask DataFrame
    return dd.read_parquet(valid_files, engine='pyarrow')

def filter_and_merge_data(volume_ddf, price_ddf, output_dir):
    """
    Filter data for 2017-2018 and merge volume and price data.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    print("Converting date columns to datetime format...")
    start_time = time.time()
    
    # Convert SETTLEMENTDATE to datetime
    volume_ddf = volume_ddf.assign(
        SETTLEMENTDATE=dd.to_datetime(volume_ddf["SETTLEMENTDATE"], errors="coerce")
    )
    price_ddf = price_ddf.assign(
        SETTLEMENTDATE=dd.to_datetime(price_ddf["SETTLEMENTDATE"], errors="coerce")
    )
    
    print(f"Date conversion defined in {time.time() - start_time:.2f} seconds")
    
    # Filter for 2017-2018 data
    print("Filtering for 2017-2018 data...")
    start_time = time.time()
    
    start_date = pd.Timestamp('2017-01-01')
    end_date = pd.Timestamp('2018-12-31 23:59:59')
    
    volume_filtered = volume_ddf[
        (volume_ddf['SETTLEMENTDATE'] >= start_date) & 
        (volume_ddf['SETTLEMENTDATE'] <= end_date)
    ]
    
    price_filtered = price_ddf[
        (price_ddf['SETTLEMENTDATE'] >= start_date) & 
        (price_ddf['SETTLEMENTDATE'] <= end_date)
    ]
    
    print(f"Filtering defined in {time.time() - start_time:.2f} seconds")
    
    # Persist the filtered DataFrames to avoid recomputation
    print("Persisting filtered data (this may take some time)...")
    start_time = time.time()
    
    volume_filtered = volume_filtered.persist()
    price_filtered = price_filtered.persist()
    
    # Wait for persist to complete
    vol_npartitions = volume_filtered.npartitions
    price_npartitions = price_filtered.npartitions
    
    print(f"Filtered volume data: {vol_npartitions} partitions")
    print(f"Filtered price data: {price_npartitions} partitions")
    print(f"Persistence completed in {time.time() - start_time:.2f} seconds")
    
    # Process one partition at a time
    for i in range(vol_npartitions):
        print(f"Processing volume partition {i+1}/{vol_npartitions}...")
        start_time = time.time()
        
        try:
            # Get one volume partition as pandas DataFrame
            vol_part = volume_filtered.get_partition(i).compute()
            print(f"Loaded volume partition with {len(vol_part)} rows in {time.time() - start_time:.2f} seconds")
            
            # Process each price partition
            for j in range(price_npartitions):
                print(f"  Processing price partition {j+1}/{price_npartitions}...")
                part_start = time.time()
                
                try:
                    # Get one price partition as pandas DataFrame
                    price_part = price_filtered.get_partition(j).compute()
                    print(f"  Loaded price partition with {len(price_part)} rows in {time.time() - part_start:.2f} seconds")
                    
                    # Merge the DataFrames
                    merge_start = time.time()
                    merged_df = vol_part.merge(
                        price_part,
                        on=["SETTLEMENTDATE", "DUID", "BIDTYPE", "BIDBAND"],
                        how="inner",
                        suffixes=('_volume', '_price')
                    )
                    print(f"  Merged to {len(merged_df)} rows in {time.time() - merge_start:.2f} seconds")
                    
                    # If we have merged data, write it to a parquet file
                    if not merged_df.empty:
                        save_start = time.time()
                        output_file = os.path.join(output_dir, f"merged_2017_2018_v{i}_p{j}.parquet")
                        merged_df.to_parquet(output_file, engine='pyarrow', index=False)
                        print(f"  Wrote {len(merged_df)} rows to {output_file} in {time.time() - save_start:.2f} seconds")
                    else:
                        print(f"  No matching data between these partitions")
                    
                except Exception as e:
                    print(f"  Error processing price partition {j}: {str(e)}")
                    continue
                
        except Exception as e:
            print(f"Error processing volume partition {i}: {str(e)}")
            continue

def main():
    # Set up paths for input files
    volume_pattern = "/Volumes/T7/bid-volume-melted-files-A4/*.parquet"
    price_pattern = "/Volumes/T7/bid-price-melted-files-B3/*.parquet"
    
    print("Loading volume data metadata...")
    volume_ddf = load_valid_parquet_files(volume_pattern)
    print(f"Volume data columns: {list(volume_ddf.columns)}")
    print(f"Volume data partitions: {volume_ddf.npartitions}")
    
    print("Loading price data metadata...")
    price_ddf = load_valid_parquet_files(price_pattern)
    print(f"Price data columns: {list(price_ddf.columns)}")
    print(f"Price data partitions: {price_ddf.npartitions}")
    
    # Create output directory if it doesn't exist
    output_dir = "/Volumes/T7/bid-merged-fcas-2017-2018"
    print(f"Will save merged data to {output_dir}...")
    
    # Filter and merge data
    filter_and_merge_data(volume_ddf, price_ddf, output_dir)
    
    # Report completion
    print("All done!")
    print(f"Output saved to: {output_dir}")

if __name__ == "__main__":
    main()